# InChI → RDKit 3D → CREST 种子（XYZ）

本笔记本将 InChI 转为 RDKit 分子，生成 3D 构象并输出 CREST 可用的 `.xyz` 种子文件。


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdmolfiles import MolToXYZFile

import os


## 输入参数
- `inchi`: 分子 InChI 字符串
- `out_xyz`: 输出的 CREST 种子 XYZ 文件名


In [ ]:
inchi = "InChI=1S/CH4/h1H4"  # 示例：甲烷
out_xyz = "seed.xyz"


## 将 InChI 转为 RDKit 分子
- 生成分子对象
- 加氢以便生成合理三维构象


In [ ]:
mol = Chem.MolFromInchi(inchi)
if mol is None:
    raise ValueError("InChI 解析失败，请检查输入。")

mol = Chem.AddHs(mol)


## 生成三维构象
- 使用 ETKDG 进行构象嵌入
- 采用 UFF 几何优化


In [ ]:
params = AllChem.ETKDGv3()
params.randomSeed = 42  # 保证可复现
res = AllChem.EmbedMolecule(mol, params)

if res != 0:
    raise RuntimeError("3D 构象生成失败。")

AllChem.UFFOptimizeMolecule(mol)


## 输出 XYZ 文件
CREST 可直接使用该文件作为种子结构：


In [ ]:
MolToXYZFile(mol, out_xyz)
print(f"已写出: {out_xyz}")


## 可选：打印 XYZ 内容检查


In [ ]:
with open(out_xyz, "r") as f:
    print(f.read())
